# Experiment: Seqrec Constrained Results Analysis

目标：
- 自动扫描 `results/test/seqrec-constrained/**/results.json`
- 聚合 `mean_results` 中的指标
- 按 `INDEX_TAG`/模型版本做可视化比较


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

DEFAULT_RESULTS_ROOT = Path(
    "/mnt/dolphinfs/hdd_pool/docker/user/hadoop-hmart-poistar/fanghaotian/GRec/results/test/seqrec-constrained"
)
RESULTS_ROOT = DEFAULT_RESULTS_ROOT

print(f"Using RESULTS_ROOT: {RESULTS_ROOT}")
print(f"Seaborn available: {HAS_SEABORN}")


## 1) 扫描并读取结果文件

如果你在其他机器运行，只需要改上面的 `RESULTS_ROOT`。

In [ ]:
result_files = sorted(RESULTS_ROOT.rglob("results.json"))
print(f"Found {len(result_files)} results.json files.")
for path in result_files[:5]:
    print(" -", path)
if len(result_files) > 5:
    print(" ...")

if not result_files:
    raise FileNotFoundError(f"No results.json found under: {RESULTS_ROOT}")


In [ ]:
def parse_results_file(path: Path, root: Path) -> dict[str, Any]:
    rel = path.relative_to(root)
    parts = rel.parts

    payload = json.loads(path.read_text(encoding="utf-8"))
    return {
        "path": str(path),
        "index_tag": parts[0] if len(parts) >= 1 else "",
        "model_dir": parts[1] if len(parts) >= 2 else "",
        "checkpoint": parts[2] if len(parts) >= 3 else "",
        "eval_split": payload.get("eval_split"),
        "rollout_cached": payload.get("rollout_cached"),
        "test_prompt_ids": payload.get("test_prompt_ids"),
        "mean_results": payload.get("mean_results", {}) or {},
        "min_results": payload.get("min_results", {}) or {},
        "max_results": payload.get("max_results", {}) or {},
    }


records = [parse_results_file(path, RESULTS_ROOT) for path in result_files]
runs_df = pd.DataFrame(records)
runs_df["file_mtime"] = runs_df["path"].map(lambda p: Path(p).stat().st_mtime)
runs_df["run_label"] = runs_df["model_dir"].fillna("") + "/" + runs_df["checkpoint"].fillna("")

print(f"Loaded runs: {len(runs_df)}")
runs_df.head(3)


## 2) 指标展开与统计

把 `mean_results` 展平成 long-format，方便聚合和画图。

In [ ]:
metric_rows = []
for _, row in runs_df.iterrows():
    mean_results = row["mean_results"] if isinstance(row["mean_results"], dict) else {}
    for metric, value in mean_results.items():
        try:
            value = float(value)
        except (TypeError, ValueError):
            continue
        metric_rows.append(
            {
                "index_tag": row["index_tag"],
                "model_dir": row["model_dir"],
                "checkpoint": row["checkpoint"],
                "run_label": row["run_label"],
                "file_mtime": row["file_mtime"],
                "metric": metric,
                "value": value,
                "path": row["path"],
            }
        )

metrics_df = pd.DataFrame(metric_rows)
if metrics_df.empty:
    raise ValueError("No metrics extracted from results.json")

print(f"Extracted metric rows: {len(metrics_df)}")
metrics_df.head(10)


In [ ]:
summary_df = (
    metrics_df.groupby(["index_tag", "metric"], as_index=False)
    .agg(
        mean_value=("value", "mean"),
        max_value=("value", "max"),
        min_value=("value", "min"),
        runs=("value", "count"),
    )
    .sort_values(["metric", "mean_value"], ascending=[True, False])
)
summary_df.head(30)


## 3) 可视化

默认取每个 `index_tag + metric` 的**最新一次**结果（按文件修改时间）。

In [ ]:
latest_metrics_df = (
    metrics_df.sort_values("file_mtime")
    .drop_duplicates(["index_tag", "metric"], keep="last")
)

preferred_metrics = ["hit@1", "hit@10", "hit@50", "ndcg@10", "ndcg@50"]
available_metrics = [m for m in preferred_metrics if m in latest_metrics_df["metric"].unique()]
if not available_metrics:
    available_metrics = sorted(latest_metrics_df["metric"].unique())[:5]

plot_df = latest_metrics_df[latest_metrics_df["metric"].isin(available_metrics)].copy()
pivot_bar = plot_df.pivot(index="index_tag", columns="metric", values="value").sort_index()

ax = pivot_bar.plot(kind="bar", figsize=(14, 6))
ax.set_title("Latest Metrics by INDEX_TAG")
ax.set_xlabel("INDEX_TAG")
ax.set_ylabel("Metric Value")
ax.legend(title="metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

pivot_bar


In [ ]:
heatmap_df = (
    latest_metrics_df.pivot(index="index_tag", columns="metric", values="value")
    .sort_index()
)

fig_w = max(10, 0.8 * len(heatmap_df.columns))
fig_h = max(4, 0.5 * len(heatmap_df.index))
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

if HAS_SEABORN:
    sns.heatmap(heatmap_df, annot=True, fmt=".4f", cmap="YlGnBu", ax=ax)
else:
    im = ax.imshow(heatmap_df.values, aspect="auto", cmap="YlGnBu")
    ax.set_xticks(range(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index)
    fig.colorbar(im, ax=ax)

ax.set_title("Latest Metrics Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
target_index = sorted(metrics_df["index_tag"].dropna().unique())[0]
focus_metrics = [m for m in ["hit@10", "ndcg@10"] if m in metrics_df["metric"].unique()]
if not focus_metrics:
    focus_metrics = sorted(metrics_df["metric"].unique())[:2]

focus_df = metrics_df[
    (metrics_df["index_tag"] == target_index) & (metrics_df["metric"].isin(focus_metrics))
].copy()
focus_df = focus_df.sort_values("file_mtime")

if focus_df.empty:
    print(f"No focus data for index_tag={target_index}")
else:
    focus_df["short_run"] = focus_df["run_label"].str[-90:]
    pivot_line = focus_df.pivot_table(
        index="short_run", columns="metric", values="value", aggfunc="mean"
    )
    ax = pivot_line.plot(marker="o", figsize=(14, 5))
    ax.set_title(f"Run Trend ({target_index})")
    ax.set_xlabel("run")
    ax.set_ylabel("metric")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    pivot_line


## 4) 导出汇总表

导出 CSV 到 `.../seqrec-constrained/_analysis/`，方便后续汇报或复用。

In [ ]:
analysis_dir = RESULTS_ROOT / "_analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

runs_export_df = runs_df.drop(columns=["mean_results", "min_results", "max_results"])
runs_out = analysis_dir / "runs_summary.csv"
metrics_out = analysis_dir / "metrics_long.csv"
latest_out = analysis_dir / "metrics_latest.csv"
summary_out = analysis_dir / "metrics_summary.csv"

runs_export_df.to_csv(runs_out, index=False)
metrics_df.to_csv(metrics_out, index=False)
latest_metrics_df.to_csv(latest_out, index=False)
summary_df.to_csv(summary_out, index=False)

print("Exported:")
print(" -", runs_out)
print(" -", metrics_out)
print(" -", latest_out)
print(" -", summary_out)
